In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.architecture import VIT_SMALL, with_architecture_results_dir
from src.baseline_config import BASELINE_CONFIG, build_training_config
from src.phase1.config import PHASE1_IMAGES_DIR, PHASE1_RUNS, PHASE1_TRAIN_CSV
from src.phase2.config import (
    PHASE2_CONFIDENCE_ONLY_FRAMES_DIR,
    PHASE2_CONFIDENCE_ONLY_THRESHOLD,
    PHASE2_FRAMES_DIR,
    PHASE2_FULL_TRAIN_CSV,
)
from src.phase2.experiment import phase2_confidence_only_csv_path
from src.phase3.config import PHASE3_RUNS
from utils.common import RESULTS_DIR
from utils.constants import VALIDATION_CSV, VALIDATION_IMAGES_DIR
from utils.metrics import (
    METRIC_KEY_SEPARATOR,
    SUMMARY_COLUMNS,
    SUMMARY_ROWS,
    collect_results_metrics,
    summarize_general_results_metrics,
)


ARCHITECTURE_CONFIG = BASELINE_CONFIG
VIT_ARCHITECTURE_CONFIG = build_training_config(VIT_SMALL)
CACHE_PATH = RESULTS_DIR / "final_lineage_validation_test_metrics.csv"
CLASS_CACHE_PATH = RESULTS_DIR / "final_lineage_validation_test_metrics_by_class.csv"
VIT_CACHE_PATH = RESULTS_DIR / "final_lineage_validation_test_metrics_vitsmall.csv"
VIT_CLASS_CACHE_PATH = RESULTS_DIR / "final_lineage_validation_test_metrics_by_class_vitsmall.csv"
FORCE_RECOMPUTE = False

TEST_CSV = Path("test/external_test.csv")
TEST_IMAGES_DIR = Path("test/images_cropped")


In [ ]:
@dataclass(frozen=True)
class ExperimentNode:
    key: str
    label: str
    phase: str
    x: float
    y_offset: float
    train_csv: Path
    images_dir: Path
    results_dirs: tuple[Path, ...]
    random_states: tuple[int, ...]
    parent: str | None = None


def phase1_results_dirs() -> tuple[Path, ...]:
    return tuple(Path(run["results_dir"]) for run in PHASE1_RUNS)


def phase1_random_states() -> tuple[int, ...]:
    return tuple(int(run["random_state"]) for run in PHASE1_RUNS)


def phase3_results_dirs(descriptor: str) -> tuple[Path, ...]:
    return tuple(Path("phase3") / descriptor / run["seed_name"] for run in PHASE3_RUNS)


def phase3_random_states() -> tuple[int, ...]:
    return tuple(int(run["random_state"]) for run in PHASE3_RUNS)


def seed_dirs(root: Path) -> tuple[Path, ...]:
    return tuple(root / f"seed_{idx}" for idx in range(1, 5))


SEED_RANDOM_STATES = (42, 123, 456, 789)
CONF060_CSV = phase2_confidence_only_csv_path(PHASE2_CONFIDENCE_ONLY_THRESHOLD)

NODES = [
    ExperimentNode(
        key="phase1",
        label="F1 base",
        phase="Fase 1",
        x=1.0,
        y_offset=0.00,
        train_csv=PHASE1_TRAIN_CSV,
        images_dir=PHASE1_IMAGES_DIR,
        results_dirs=phase1_results_dirs(),
        random_states=phase1_random_states(),
    ),
    ExperimentNode(
        key="train",
        label="train",
        phase="Fase 2",
        x=2.0,
        y_offset=-0.28,
        train_csv=PHASE2_FULL_TRAIN_CSV,
        images_dir=PHASE2_FRAMES_DIR,
        results_dirs=seed_dirs(Path("phase2/train")),
        random_states=SEED_RANDOM_STATES,
        parent="phase1",
    ),
    ExperimentNode(
        key="conf060",
        label="conf060",
        phase="Fase 2",
        x=2.0,
        y_offset=0.28,
        train_csv=CONF060_CSV,
        images_dir=PHASE2_CONFIDENCE_ONLY_FRAMES_DIR,
        results_dirs=seed_dirs(Path("phase2/conf060")),
        random_states=SEED_RANDOM_STATES,
        parent="phase1",
    ),
    ExperimentNode(
        key="train_conf040",
        label="train_conf040",
        phase="Fase 3",
        x=3.0,
        y_offset=-0.30,
        train_csv=Path("phase3/phase3_train_conf040.csv"),
        images_dir=PHASE2_FRAMES_DIR,
        results_dirs=seed_dirs(Path("phase3/train_conf040")),
        random_states=SEED_RANDOM_STATES,
        parent="train",
    ),
    ExperimentNode(
        key="train_conf040_dedup_p75_25",
        label="train_conf040 dedup p75/25",
        phase="Fase 3",
        x=3.0,
        y_offset=0.00,
        train_csv=Path("phase3/phase3_train_conf040_dedup_p75_25.csv"),
        images_dir=PHASE2_FRAMES_DIR,
        results_dirs=phase3_results_dirs("train_conf040_dedup_p75_25"),
        random_states=phase3_random_states(),
        parent="train_conf040",
    ),
    ExperimentNode(
        key="conf060_dedup_p75_25",
        label="conf060 dedup p75/25",
        phase="Fase 3",
        x=3.0,
        y_offset=0.30,
        train_csv=Path("phase3/phase3_conf060_dedup_p75_25.csv"),
        images_dir=PHASE2_CONFIDENCE_ONLY_FRAMES_DIR,
        results_dirs=phase3_results_dirs("conf060_dedup_p75_25"),
        random_states=phase3_random_states(),
        parent="conf060",
    ),
]

pd.DataFrame(
    [
        {
            "key": node.key,
            "phase": node.phase,
            "parent": node.parent,
            "train_csv": str(node.train_csv),
            "images_dir": str(node.images_dir),
            "results_dirs": ", ".join(str(path) for path in node.results_dirs),
        }
        for node in NODES
    ]
)


In [ ]:
def empty_node_split_summary(node: ExperimentNode, split: str) -> dict:
    return {
        "key": node.key,
        "label": node.label,
        "phase": node.phase,
        "parent": node.parent,
        "split": split,
        "n_seeds": 0,
        "macro_f1": np.nan,
        "macro_f1_std": np.nan,
        "mcc": np.nan,
        "accuracy": np.nan,
        "precision": np.nan,
        "recall": np.nan,
    }


def empty_node_split_class_summaries(node: ExperimentNode, split: str) -> list[dict]:
    rows = []
    for scope in SUMMARY_ROWS:
        row = {
            "key": node.key,
            "label": node.label,
            "phase": node.phase,
            "parent": node.parent,
            "split": split,
            "scope": scope,
            "n_seeds": 0,
        }
        for metric_name in SUMMARY_COLUMNS:
            row[metric_name] = np.nan
            row[f"{metric_name}_std"] = np.nan
        rows.append(row)
    return rows


def summarize_scoped_results_metrics(
    per_seed_metrics: pd.DataFrame,
    metadata: dict,
) -> list[dict]:
    rows = []
    for scope in SUMMARY_ROWS:
        row = {**metadata, "scope": scope}
        for metric_name in SUMMARY_COLUMNS:
            metric_key = f"{scope}{METRIC_KEY_SEPARATOR}{metric_name}"
            values = pd.to_numeric(per_seed_metrics[metric_key], errors="raise")
            row[metric_name] = float(values.mean())
            row[f"{metric_name}_std"] = (
                float(values.std(ddof=1)) if len(values) > 1 else np.nan
            )
        rows.append(row)
    return rows


def collect_node_split_summaries(
    node: ExperimentNode,
    split: str,
    csv_path: Path,
    images_dir: Path,
    architecture_config,
) -> tuple[dict, list[dict]]:
    existing_results_dirs = [
        results_dir
        for results_dir in node.results_dirs
        if (
            RESULTS_DIR
            / with_architecture_results_dir(architecture_config.architecture, results_dir)
            / "best_baseline_model.pth"
        ).exists()
    ]
    if not existing_results_dirs:
        return (
            empty_node_split_summary(node, split),
            empty_node_split_class_summaries(node, split),
        )

    random_states = node.random_states[: len(existing_results_dirs)]
    per_seed = collect_results_metrics(
        results_dirs=existing_results_dirs,
        validation_csv_dir=csv_path,
        validation_img_dir=images_dir,
        training_config=architecture_config,
        random_states=random_states,
    )
    summary = summarize_general_results_metrics(per_seed)
    metadata = {
        "key": node.key,
        "label": node.label,
        "phase": node.phase,
        "parent": node.parent,
        "split": split,
        "n_seeds": len(existing_results_dirs),
    }
    general_summary = {
        **metadata,
        "macro_f1": summary["macro_f1"],
        "macro_f1_std": summary["macro_f1_std"],
        "mcc": summary["mcc"],
        "accuracy": summary["accuracy"],
        "precision": summary["precision"],
        "recall": summary["recall"],
    }
    return general_summary, summarize_scoped_results_metrics(per_seed, metadata)


def compute_lineage_metrics(architecture_config) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    class_rows = []
    for node in NODES:
        for split, csv_path, images_dir in [
            ("validation", VALIDATION_CSV, VALIDATION_IMAGES_DIR),
            ("test", TEST_CSV, TEST_IMAGES_DIR),
        ]:
            summary, scoped_summaries = collect_node_split_summaries(
                node=node,
                split=split,
                csv_path=csv_path,
                images_dir=images_dir,
                architecture_config=architecture_config,
            )
            rows.append(summary)
            class_rows.extend(scoped_summaries)
    return pd.DataFrame(rows), pd.DataFrame(class_rows)


if (
    CACHE_PATH.exists()
    and CLASS_CACHE_PATH.exists()
    and not FORCE_RECOMPUTE
):
    metrics_df = pd.read_csv(CACHE_PATH)
    class_metrics_df = pd.read_csv(CLASS_CACHE_PATH)
else:
    metrics_df, class_metrics_df = compute_lineage_metrics(ARCHITECTURE_CONFIG)
    CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    metrics_df.to_csv(CACHE_PATH, index=False)
    class_metrics_df.to_csv(CLASS_CACHE_PATH, index=False)

if (
    VIT_CACHE_PATH.exists()
    and VIT_CLASS_CACHE_PATH.exists()
    and not FORCE_RECOMPUTE
):
    vit_metrics_df = pd.read_csv(VIT_CACHE_PATH)
    vit_class_metrics_df = pd.read_csv(VIT_CLASS_CACHE_PATH)
else:
    vit_metrics_df, vit_class_metrics_df = compute_lineage_metrics(VIT_ARCHITECTURE_CONFIG)
    VIT_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    vit_metrics_df.to_csv(VIT_CACHE_PATH, index=False)
    vit_class_metrics_df.to_csv(VIT_CLASS_CACHE_PATH, index=False)

metrics_df.sort_values(["split", "macro_f1"], ascending=[True, False])


In [ ]:
summary_table = (
    metrics_df
    .assign(parent=lambda df: df["parent"].fillna("-"))
    .pivot_table(
        index=["phase", "label", "parent", "n_seeds"],
        columns="split",
        values=["macro_f1", "macro_f1_std", "mcc", "accuracy", "precision", "recall"],
        aggfunc="first",
    )
    .sort_values(("macro_f1", "test"), ascending=False)
)

summary_table


In [ ]:
PRESENTATION_LABELS = {
    "phase1": "Modelo base",
    "train": "Ampliación balanceada",
    "conf060": "Ampliación por confianza",
    "train_conf040": "Filtrado por confianza",
    "train_conf040_dedup_p75_25": "Curación temporal",
    "conf060_dedup_p75_25": "Curación temporal final",
}

HIGHLIGHT_VALIDATION_POINTS = {
    "EfficientNet": {
        "key": "conf060",
        "label": "EfficientNet-B0: 0,586",
    },
    "ViT-Small": {
        "key": "conf060_dedup_p75_25",
        "label": "ViT-Small: 0,623",
    },
}


def format_decimal_es(value: float) -> str:
    return f"{value:.3f}".replace(".", ",")


def plot_architecture_lineage(
    data: pd.DataFrame,
    architectures: tuple[str, ...],
    title: str,
    ax=None,
):
    node_by_key = {node.key: node for node in NODES}
    plot_nodes = [node for node in NODES]
    if ax is None:
        _, ax = plt.subplots(figsize=(13, 7))

    phase_bands = [
        (0.65, 1.45, "Fase 1 · Modelo base", "#f8fafc"),
        (1.55, 2.45, "Fase 2 · Ampliación desde vídeo", "#f1f5f9"),
        (2.55, 3.62, "Fase 3 · Curación de datos", "#f8fafc"),
    ]
    for xmin, xmax, label, color in phase_bands:
        ax.axvspan(xmin, xmax, color=color, zorder=0)
        ax.text(
            (xmin + xmax) / 2,
            0.995,
            label,
            transform=ax.get_xaxis_transform(),
            ha="center",
            va="top",
            fontsize=9,
            fontweight="bold",
            color="#334155",
        )

    architecture_styles = {
        "EfficientNet": {"offset": -0.055, "marker": "o", "alpha": 0.80},
        "ViT-Small": {"offset": 0.055, "marker": "^", "alpha": 0.80},
    }
    split_styles = {
        "validation": {
            "color": "#2563eb",
            "linestyle": "--",
            "label": "Validation",
            "offset": -0.022,
            "label_y": -15,
        },
        "test": {
            "color": "#dc2626",
            "linestyle": "--",
            "label": "Test externo",
            "offset": 0.022,
            "label_y": 11,
        },
    }

    plotted_points = {}
    for architecture in architectures:
        arch_style = architecture_styles[architecture]
        architecture_df = data[data["architecture"].eq(architecture)]
        metric_by_key_split = {
            (row.key, row.split): row
            for row in architecture_df.itertuples(index=False)
        }

        for split, split_style in split_styles.items():
            legend_label = f"{architecture} - {split_style['label']}"
            for node_index, node in enumerate(plot_nodes):
                row = metric_by_key_split.get((node.key, split))
                if row is None or pd.isna(row.macro_f1):
                    continue
                x = node.x + node.y_offset + arch_style["offset"] + split_style["offset"]
                yerr = 0 if pd.isna(row.macro_f1_std) else row.macro_f1_std
                ax.errorbar(
                    x,
                    row.macro_f1,
                    yerr=yerr,
                    fmt=arch_style["marker"],
                    color=split_style["color"],
                    ecolor=split_style["color"],
                    elinewidth=1.2,
                    capsize=3,
                    markersize=7,
                    alpha=arch_style["alpha"],
                    label=legend_label if node_index == 0 else None,
                    zorder=3,
                )
                plotted_points[(architecture, node.key, split)] = (x, row.macro_f1)
                ax.annotate(
                    format_decimal_es(row.macro_f1),
                    xy=(x, row.macro_f1),
                    xytext=(0, split_style["label_y"]),
                    textcoords="offset points",
                    ha="center",
                    va="center",
                    fontsize=8,
                    color=split_style["color"],
                )

            for node in plot_nodes:
                if node.parent is None:
                    continue
                parent = node_by_key[node.parent]
                parent_row = metric_by_key_split.get((parent.key, split))
                child_row = metric_by_key_split.get((node.key, split))
                if parent_row is None or child_row is None:
                    continue
                if pd.isna(parent_row.macro_f1) or pd.isna(child_row.macro_f1):
                    continue
                ax.plot(
                    [
                        parent.x + parent.y_offset + arch_style["offset"] + split_style["offset"],
                        node.x + node.y_offset + arch_style["offset"] + split_style["offset"],
                    ],
                    [parent_row.macro_f1, child_row.macro_f1],
                    color=split_style["color"],
                    linestyle=split_style["linestyle"],
                    linewidth=1.1,
                    alpha=0.22,
                    zorder=2,
                )

        highlight = HIGHLIGHT_VALIDATION_POINTS.get(architecture)
        if highlight is not None:
            point = plotted_points.get((architecture, highlight["key"], "validation"))
            if point is not None:
                x, y = point
                ax.scatter(
                    [x],
                    [y],
                    s=115,
                    facecolors="none",
                    edgecolors="#16a34a",
                    linewidths=2,
                    zorder=4,
                )
                ax.annotate(
                    highlight["label"],
                    xy=(x, y),
                    xytext=(18, 20),
                    textcoords="offset points",
                    ha="left",
                    va="bottom",
                    fontsize=9,
                    fontweight="bold",
                    color="#15803d",
                    arrowprops={
                        "arrowstyle": "->",
                        "color": "#16a34a",
                        "linewidth": 1.2,
                    },
                )

    for node in plot_nodes:
        node_rows = data[data["key"].eq(node.key)]
        if node_rows["macro_f1"].dropna().empty:
            continue
        ax.annotate(
            PRESENTATION_LABELS.get(node.key, node.label),
            xy=(node.x + node.y_offset, 0.94),
            xycoords=("data", "axes fraction"),
            xytext=(0, -4),
            textcoords="offset points",
            ha="center",
            va="top",
            fontsize=8.5,
            rotation=30,
            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.75, "pad": 1.5},
            clip_on=False,
        )

    ax.set_xlim(0.65, 3.62)
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels([])
    ax.tick_params(axis="x", length=0)
    ax.set_ylabel("Macro-F1")
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.25)
    ax.legend(loc="lower right", fontsize=9)
    ax.margins(x=0.08)

    valid_f1 = data["macro_f1"].dropna()
    if not valid_f1.empty:
        margin = max(0.015, float(valid_f1.std()) * 0.75)
        ax.set_ylim(
            max(0, float(valid_f1.min()) - margin),
            min(1, float(valid_f1.max()) + margin * 1.8),
        )
    return ax


In [ ]:
efficientnet_lineage_df = (
    metrics_df.copy()
    if "metrics_df" in globals()
    else pd.read_csv(CACHE_PATH)
).assign(architecture="EfficientNet")
vit_lineage_df = (
    vit_metrics_df.copy()
    if "vit_metrics_df" in globals()
    else pd.read_csv(VIT_CACHE_PATH)
).assign(architecture="ViT-Small")
multi_arch_lineage_df = pd.concat(
    [efficientnet_lineage_df, vit_lineage_df],
    ignore_index=True,
)

fig, axes = plt.subplots(1, 2, figsize=(17, 6), sharey=True)
plot_architecture_lineage(
    multi_arch_lineage_df[multi_arch_lineage_df["architecture"].eq("EfficientNet")],
    architectures=("EfficientNet",),
    title="EfficientNet-B0",
    ax=axes[0],
)
plot_architecture_lineage(
    multi_arch_lineage_df[multi_arch_lineage_df["architecture"].eq("ViT-Small")],
    architectures=("ViT-Small",),
    title="ViT-Small",
    ax=axes[1],
)

fig.text(
    0.5,
    0.02,
    "Macro-F1 media ± desviación estándar sobre 4 semillas.",
    ha="center",
    va="bottom",
    fontsize=10,
    color="#334155",
)
plt.tight_layout(rect=(0, 0.06, 1, 1))
plt.show()


In [ ]:
from collections.abc import Iterable

import timm
import torch
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms


ORIGINAL_IMAGES_DIR = Path("data/images_cropped")
EXTERNAL_TEST_IMAGES_DIR = Path("data/test/images_cropped")
TSNE_EMBEDDINGS_PATH = RESULTS_DIR / "cropped_main_external_test_efficientnet_embeddings.npz"
TSNE_COORDINATES_PATH = RESULTS_DIR / "cropped_main_external_test_tsne.csv"
TSNE_RANDOM_STATE = 42
FORCE_TSNE_RECOMPUTE = False
MAX_IMAGES_PER_GROUP: int | None = None
IMAGE_EXTENSIONS = {".bmp", ".jpeg", ".jpg", ".png", ".tif", ".tiff", ".webp"}
MAIN_LABEL = "Conjunto principal (Hospital Clinic)"
EXTERNAL_TEST_LABEL = "Test externo"


def image_paths(images_dir: Path, max_images: int | None) -> list[Path]:
    paths = sorted(
        path
        for path in images_dir.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )
    if not paths:
        raise FileNotFoundError(f"No images found in {images_dir}")
    return paths if max_images is None else paths[:max_images]


class ImagePathsDataset(Dataset):
    def __init__(self, paths: Iterable[Path], transform):
        self.paths = list(paths)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, index: int):
        with Image.open(self.paths[index]) as image:
            return self.transform(image.convert("RGB"))


def extract_efficientnet_embeddings(paths: list[Path]) -> np.ndarray:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    transform = transforms.Compose(
        [
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225),
            ),
        ]
    )
    loader = DataLoader(
        ImagePathsDataset(paths, transform),
        batch_size=64 if device.type == "cuda" else 16,
        shuffle=False,
        num_workers=0,
    )
    model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=0)
    model.to(device).eval()

    batches = []
    with torch.inference_mode():
        for batch in loader:
            batches.append(model(batch.to(device)).cpu().numpy())
    return np.concatenate(batches, axis=0)


original_paths = image_paths(ORIGINAL_IMAGES_DIR, MAX_IMAGES_PER_GROUP)
external_test_paths = image_paths(EXTERNAL_TEST_IMAGES_DIR, MAX_IMAGES_PER_GROUP)
all_paths = original_paths + external_test_paths
sources = np.array(
    [MAIN_LABEL] * len(original_paths) + [EXTERNAL_TEST_LABEL] * len(external_test_paths)
)

path_strings = [str(path) for path in all_paths]
embeddings = None
if TSNE_EMBEDDINGS_PATH.exists() and not FORCE_TSNE_RECOMPUTE:
    cache = np.load(TSNE_EMBEDDINGS_PATH, allow_pickle=True)
    if cache["paths"].astype(str).tolist() == path_strings:
        embeddings = cache["embeddings"]

if embeddings is None:
    embeddings = extract_efficientnet_embeddings(all_paths)
    TSNE_EMBEDDINGS_PATH.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        TSNE_EMBEDDINGS_PATH,
        embeddings=embeddings,
        paths=np.array(path_strings),
    )

if TSNE_COORDINATES_PATH.exists() and not FORCE_TSNE_RECOMPUTE:
    tsne_df = pd.read_csv(TSNE_COORDINATES_PATH)
    if tsne_df["path"].tolist() != path_strings:
        tsne_df = None
else:
    tsne_df = None

if tsne_df is None:
    pca_components = min(50, embeddings.shape[0] - 1, embeddings.shape[1])
    reduced_embeddings = PCA(n_components=pca_components, random_state=TSNE_RANDOM_STATE).fit_transform(embeddings)
    perplexity = min(30, max(5, (len(reduced_embeddings) - 1) // 3))
    coordinates = TSNE(
        n_components=2,
        perplexity=perplexity,
        init="pca",
        learning_rate="auto",
        random_state=TSNE_RANDOM_STATE,
    ).fit_transform(reduced_embeddings)
    tsne_df = pd.DataFrame(
        {
            "x": coordinates[:, 0],
            "y": coordinates[:, 1],
            "source": sources,
            "path": path_strings,
        }
    )
    tsne_df.to_csv(TSNE_COORDINATES_PATH, index=False)

fig, ax = plt.subplots(figsize=(10, 8), constrained_layout=True)
for source, color in [(MAIN_LABEL, "#2878B5"), (EXTERNAL_TEST_LABEL, "#E07B39")]:
    subset = tsne_df[tsne_df["source"] == source]
    ax.scatter(subset["x"], subset["y"], s=9, alpha=0.58, color=color, label=source, linewidths=0)
ax.set(
    title="Proyección t-SNE de representaciones visuales",
    xlabel="t-SNE 1",
    ylabel="t-SNE 2",
)
ax.legend(frameon=False, markerscale=2)
plt.show()

tsne_df.groupby("source").size().rename("n_images")
